# Project: E-Commerce Data Lakehouse
## Stage 3: Gold Layer - Business Aggregations & Analytics
This notebook transforms the cleaned Silver data into a business-ready **Gold layer**. We will aggregate sales data by customer to track performance and revenue metrics.

03_gold_transformation_and_analytics

In [0]:
%sql
USE rishikesh_db;

In [0]:
%sql
-- 2. Create a temporary view from your actual Silver table
CREATE OR REPLACE TEMPORARY VIEW silver_sales_view AS
SELECT * FROM silver_sales_enriched;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_sales_summary
USING DELTA
AS
SELECT 
    customer_name,
    date_trunc('DD', silver_load_time) AS sale_date,
    SUM(sale_amount) AS total_revenue,
    COUNT(order_id) AS total_orders
FROM silver_sales_enriched
GROUP BY customer_name, sale_date
ORDER BY total_revenue DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold_sales_summary
LIMIT 20;

customer_name,sale_date,total_revenue,total_orders
digital lifestyle solutions,2026-01-30T00:00:00.000Z,10083.50,67
"atlanta digital studio, llc",2026-01-30T00:00:00.000Z,9632.00,64
statusdigital,2026-01-30T00:00:00.000Z,9632.00,64
genesis electronics recycling,2026-01-30T00:00:00.000Z,9632.00,64
rpm optoelectronics,2026-01-30T00:00:00.000Z,9481.50,63
digital photo solutions ltd,2026-01-30T00:00:00.000Z,9331.00,62
digital attic,2026-01-30T00:00:00.000Z,9030.00,60
"bradsworth digital solutions, inc",2026-01-30T00:00:00.000Z,8879.50,59
modern digital imaging,2026-01-30T00:00:00.000Z,8729.00,58
pro digital solutions,2026-01-30T00:00:00.000Z,8578.50,57


In [0]:
%sql
-- 1. Optimize the table for faster filtering by customer
OPTIMIZE gold_sales_summary 
ZORDER BY (customer_name);

-- 2. Create a reporting view (hiding technical load times)
CREATE OR REPLACE VIEW v_executive_sales_dashboard AS
SELECT 
    customer_name AS Customer,
    sale_date AS Order_Date,
    total_revenue AS Revenue,
    total_orders AS Units_Sold
FROM gold_sales_summary;